# Downgrade quality: real d10 template case

This notebook isolates the real `d10` map analysis from the full demo. It compares `ud_grade`, `harmonic_ud_grade`, and `smoothing + ud_grade` using residual-map and spectral diagnostics.

### Important scope note

- Unlike the synthetic notebook, there is no independent pixel-level truth map at target NSIDE.
- Therefore, diagnostics here are comparative (method-vs-method), not absolute error-to-truth metrics.


In [ ]:
from pathlib import Path
import urllib.request

import numpy as np
import healpy as hp
import matplotlib.pyplot as plt


In [ ]:
def _round_sig(x, sig=2):
    if x == 0 or not np.isfinite(x):
        return float(x)
    return float(np.round(x, sig - int(np.floor(np.log10(abs(x)))) - 1))

def rounded_limits(*maps, q=(0.5, 99.5)):
    vals = np.concatenate([m[np.isfinite(m)] for m in maps])
    lo, hi = np.percentile(vals, q)
    vmin = _round_sig(float(lo), sig=2)
    vmax = _round_sig(float(hi), sig=2)
    if vmin == vmax:
        vmax = vmin + 1.0
    return vmin, vmax

def proj_panels(maps, titles, cmap='RdBu_r', q=(0.5, 99.5), ncol=3, xsize=2200, symmetric=False):
    vmin, vmax = rounded_limits(*maps, q=q)
    if symmetric:
        lim = max(abs(vmin), abs(vmax))
        vmin, vmax = -lim, lim
    print(f'vmin={vmin}, vmax={vmax}')
    n = len(maps)
    ncol = min(ncol, n)
    nrow = int(np.ceil(n / ncol))
    plt.figure(figsize=(6.8*ncol, 4.8*nrow))
    for i, (m, t) in enumerate(zip(maps, titles), start=1):
        hp.projview(
            m,
            sub=(nrow, ncol, i),
            title=t,
            min=vmin,
            max=vmax,
            cmap=cmap,
            graticule=True,
            xsize=xsize,
            cb_orientation='horizontal',
        )
    plt.tight_layout()

def compare_to_ref(m, m_ref, cl, cl_ref, ell_min=2):
    good = np.isfinite(m) & np.isfinite(m_ref)
    m0 = m[good]
    r0 = m_ref[good]
    rmse_rel = np.sqrt(np.mean((m0-r0)**2)) / np.std(r0)
    mae_rel = np.mean(np.abs(m0-r0)) / np.std(r0)
    corr = np.corrcoef(m0, r0)[0, 1]
    spec_rel_l2 = np.linalg.norm(cl[ell_min:] - cl_ref[ell_min:]) / np.linalg.norm(cl_ref[ell_min:])
    return {
        'rmse_rel_std': float(rmse_rel),
        'mae_rel_std': float(mae_rel),
        'map_corr': float(corr),
        'spec_rel_l2': float(spec_rel_l2),
    }

def moll_diff_panels(diff_maps, titles, unit='', q=(0.2, 99.8)):
    vmin, vmax = rounded_limits(*diff_maps, q=q)
    lim = max(abs(vmin), abs(vmax))
    vmin, vmax = -lim, lim
    print(f'moll diff scale: vmin={vmin}, vmax={vmax}')
    n = len(diff_maps)
    fig = plt.figure(figsize=(6.8*n, 5.0))
    for i, (m, t) in enumerate(zip(diff_maps, titles), start=1):
        hp.mollview(
            m,
            fig=fig.number,
            sub=(1, n, i),
            title=t,
            min=vmin,
            max=vmax,
            cmap='RdBu_r',
            unit=unit,
            cbar=True,
        )
    plt.tight_layout()


## 1) Real d10 353 GHz intensity template (direct download, no PySM)

### Method notes for d10

- `d10` has strong Galactic-structure contrast, so downgrade artifacts may be morphologically obvious in residual maps.
- `harmonic lmax=3N-1` is used as a practical reference in residual visualizations in this notebook.
- This reference is not external truth; it is a comparison anchor with full target band-limit in harmonic space.


In [ ]:
base_url = 'https://portal.nersc.gov/project/cmb/pysm-data'
relpath = 'dust_gnilc/gnilc_dust_template_nside2048_2023.02.10.fits'
url = f'{base_url}/{relpath}'

cache_dir = Path('/tmp/pysm_data')
cache_dir.mkdir(parents=True, exist_ok=True)
local_path = cache_dir / Path(relpath).name
if not local_path.exists():
    print('Downloading', url)
    urllib.request.urlretrieve(url, local_path)
print('Local file:', local_path)

In [ ]:
m_d10 = hp.read_map(local_path, field=0, dtype=np.float64)
nside_d10 = hp.get_nside(m_d10)
print('d10 nside:', nside_d10, 'npix:', m_d10.size)

In [ ]:
nside_out_real = 128
lmax_real = 3*nside_out_real - 1

d10_ud = hp.ud_grade(m_d10, nside_out=nside_out_real)
d10_harm_default = hp.harmonic_ud_grade(m_d10, nside_out=nside_out_real)
d10_harm_lmax_out = hp.harmonic_ud_grade(m_d10, nside_out=nside_out_real, lmax=lmax_real)
d10_smooth_ud = hp.ud_grade(hp.smoothing(m_d10, fwhm=np.radians(30/60)), nside_out=nside_out_real)

In [ ]:
proj_panels(
    [d10_ud, d10_harm_default, d10_harm_lmax_out, d10_smooth_ud],
    ['d10: ud_grade', 'd10: harmonic default', 'd10: harmonic lmax=3N-1', 'd10: smoothing + ud'],
    cmap='viridis',
    q=(1, 99),
)

### d10 residual maps (visual check)
For d10, there is no independent target-NSIDE ground-truth map analogous to `m_ref` in the synthetic case.
Therefore this section uses relative diagnostics (method-to-method residual maps and spectra) rather than the scalar truth-referenced metrics used in Section 1.


In [ ]:
d10_ref = d10_harm_lmax_out
proj_panels(
    [d10_ud - d10_ref, d10_harm_default - d10_ref, d10_smooth_ud - d10_ref],
    ['d10: ud - harm(lmax=3N-1)', 'd10: harm(default) - harm(lmax=3N-1)', 'd10: smooth+ud - harm(lmax=3N-1)'],
    cmap='RdBu_r',
    q=(0.2, 99.8),
    symmetric=True,
)


### d10 Mollweide differences vs harmonic reference
Mollweide residuals use the same symmetric color range across methods.


In [ ]:
moll_diff_panels(
    [d10_ud - d10_ref, d10_harm_default - d10_ref, d10_smooth_ud - d10_ref],
    ['d10: ud - harm(lmax=3N-1)', 'd10: harm(default) - harm(lmax=3N-1)', 'd10: smooth+ud - harm(lmax=3N-1)'],
    unit='MJy/sr',
    q=(0.2, 99.8),
)


### How to read unmasked d10 spectra

- The original input spectrum is included as context for what power exists before downgrade.
- At fixed `nside_out`, methods differ mainly in how they handle near-bandlimit content.
- Compare slope and normalization changes jointly; a lower curve is not automatically better without considering residual maps and masked behavior.


In [ ]:
cl_d10_orig = hp.anafast(m_d10, lmax=lmax_real)
cl_d10_ud = hp.anafast(d10_ud, lmax=lmax_real)
cl_d10_harm_default = hp.anafast(d10_harm_default, lmax=lmax_real)
cl_d10_harm_lmax_out = hp.anafast(d10_harm_lmax_out, lmax=lmax_real)
cl_d10_smooth_ud = hp.anafast(d10_smooth_ud, lmax=lmax_real)
ell_real = np.arange(lmax_real + 1)

line_styles = {
    "original": "-",
    "ud_grade": "--",
    "harmonic_default": "-.",
    "harmonic_lmax": ":",
    "smoothing_ud": (0, (5, 2)),
}

plt.figure(figsize=(8,4.5))
plt.loglog(ell_real[2:], cl_d10_orig[2:], label="original d10 input", linestyle=line_styles["original"], linewidth=2.2)
plt.loglog(ell_real[2:], cl_d10_ud[2:], label="ud_grade", linestyle=line_styles["ud_grade"], linewidth=2.0)
plt.loglog(ell_real[2:], cl_d10_harm_default[2:], label="harmonic default", linestyle=line_styles["harmonic_default"], linewidth=2.0)
plt.loglog(ell_real[2:], cl_d10_harm_lmax_out[2:], label="harmonic lmax=3N-1", linestyle=line_styles["harmonic_lmax"], linewidth=2.4)
plt.loglog(ell_real[2:], cl_d10_smooth_ud[2:], label="smoothing + ud", linestyle=line_styles["smoothing_ud"], linewidth=2.0)
plt.xlabel("ell")
plt.ylabel("C_ell")
plt.title("d10 intensity spectra: original and downgraded maps")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()


For d10, the apparent spectral "drop" with harmonic downgrade is expected when `lmax` is conservative.
Using `lmax=3*nside_out-1` shifts the harmonic result toward the full target-bandlimit while still avoiding `ud_grade` alias leakage.

### d10 spectra with Planck Galactic mask (large cut)
We apply a Planck Galactic mask (`GAL060`, from `HFI_Mask_GalPlane-apo0_2048_R2.00`) to emphasize high-latitude behavior and reduce Galactic-plane dominance in the spectra.

- The mask reduces Galactic-plane dominance and makes high-latitude differences easier to compare.
- NaMaster decoupling is used to reduce mode-coupling bias from sky cut, improving inter-method interpretability.


In [ ]:
from astropy.io import fits
import urllib.request
from pathlib import Path
import healpy as hp
import numpy as np
import matplotlib.pyplot as plt

# Load the mask file (download if missing)
mask_filename = "HFI_Mask_GalPlane-apo0_2048_R2.00.fits"
mask_url = f"https://irsa.ipac.caltech.edu/data/Planck/release_2/ancillary-data/masks/{mask_filename}"
mask_path = Path(mask_filename)
if not mask_path.exists():
    urllib.request.urlretrieve(mask_url, mask_path)

# Large Galactic cut: GAL060 keeps ~60% of sky (apodized).
mask_field = "GAL060"
with fits.open(mask_path) as hdul:
    col_names = list(hdul[1].columns.names)
mask_field_idx = col_names.index(mask_field)
mask_hi = hp.read_map(mask_path, field=mask_field_idx, dtype=np.float64)

# Plot the mask
hp.projview(mask_hi, title=f"Planck {mask_field} mask", unit="Mask value", min=0, max=1)
hp.graticule()

### Interpreting masked, decoupled spectra

- Focus on relative ordering and shape consistency across methods rather than absolute amplitude alone.
- Persistent high-`\ell` excess suggests residual leakage/aliasing; persistent suppression suggests stronger effective smoothing.
- Combine this panel with residual maps to distinguish localized artifacts from broad transfer-function effects.


In [ ]:
import pymaster as nmt

def mask_at_nside(mask, nside_out):
    nside_in = hp.get_nside(mask)
    if nside_in == nside_out:
        out = mask
    else:
        out = hp.ud_grade(mask, nside_out=nside_out)
    return np.clip(out, 0.0, 1.0)

def nmt_decoupled_cl(map_in, mask_base, lmax, nlb=8):
    nside = hp.get_nside(map_in)
    mask = mask_at_nside(mask_base, nside)
    fsky_eff = float(np.mean(mask**2))
    field = nmt.NmtField(mask, [map_in])
    b = nmt.NmtBin.from_nside_linear(nside, nlb=nlb, is_Dell=False)
    wsp = nmt.NmtWorkspace()
    wsp.compute_coupling_matrix(field, field, b)
    cl_coupled = nmt.compute_coupled_cell(field, field)
    cl_dec = wsp.decouple_cell(cl_coupled)[0]
    ells_eff = b.get_effective_ells()
    good = ells_eff <= lmax
    return ells_eff[good], cl_dec[good], fsky_eff

# For a fair low-resolution comparison, include an original-map baseline downgraded to target NSIDE.
d10_orig_to_out = hp.ud_grade(m_d10, nside_out=nside_out_real)

ell_nmt, cl_d10_orig_nmt, fsky_out = nmt_decoupled_cl(d10_orig_to_out, mask_hi, lmax_real)
_, cl_d10_ud_nmt, _ = nmt_decoupled_cl(d10_ud, mask_hi, lmax_real)
_, cl_d10_harm_default_nmt, _ = nmt_decoupled_cl(d10_harm_default, mask_hi, lmax_real)
_, cl_d10_harm_lmax_out_nmt, _ = nmt_decoupled_cl(d10_harm_lmax_out, mask_hi, lmax_real)
_, cl_d10_smooth_ud_nmt, _ = nmt_decoupled_cl(d10_smooth_ud, mask_hi, lmax_real)

print(f"NaMaster mask field: {mask_field} (column {mask_field_idx}), f_sky_eff(nside={nside_out_real})={fsky_out:.3f}")

line_styles = {
    "original": "-",
    "ud_grade": "--",
    "harmonic_default": "-.",
    "harmonic_lmax": ":",
    "smoothing_ud": (0, (5, 2)),
}

plt.figure(figsize=(8,4.5))
plt.loglog(ell_nmt[ell_nmt>=2], cl_d10_orig_nmt[ell_nmt>=2], label="original->nside_out (masked, NaMaster)", linestyle=line_styles["original"], linewidth=2.2)
plt.loglog(ell_nmt[ell_nmt>=2], cl_d10_ud_nmt[ell_nmt>=2], label="ud_grade (masked, NaMaster)", linestyle=line_styles["ud_grade"], linewidth=2.0)
plt.loglog(ell_nmt[ell_nmt>=2], cl_d10_harm_default_nmt[ell_nmt>=2], label="harmonic default (masked, NaMaster)", linestyle=line_styles["harmonic_default"], linewidth=2.0)
plt.loglog(ell_nmt[ell_nmt>=2], cl_d10_harm_lmax_out_nmt[ell_nmt>=2], label="harmonic lmax=3N-1 (masked, NaMaster)", linestyle=line_styles["harmonic_lmax"], linewidth=2.4)
plt.loglog(ell_nmt[ell_nmt>=2], cl_d10_smooth_ud_nmt[ell_nmt>=2], label="smoothing + ud (masked, NaMaster)", linestyle=line_styles["smoothing_ud"], linewidth=2.0)
plt.xlabel("ell")
plt.ylabel("decoupled C_ell")
plt.title(f"d10 masked spectra with Planck {mask_field} mask (NaMaster decoupled)")
plt.grid(alpha=0.25)
plt.legend(fontsize=8)
plt.tight_layout()